

Load pre-extracted CLIP features, train a linear probe, evaluate it, then test with Tip-Adapter.


In [1]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from tqdm import tqdm


In [2]:
# TRAIN_FEATURE_PATH = "/kaggle/input/your-feature-dataset/ffpp_train_features.pt"
# CELEB_TEST_FEATURE_PATH = "/kaggle/input/your-feature-dataset/celebdf_test_features.pt"
# FFPP_TEST_FEATURE_PATH = "/kaggle/input/your-feature-dataset/ffpp_test_features.pt"

TRAIN_FEATURE_PATH = "/kaggle/input/datasets/vhonghoavin/deepfakebench-features/ffpp_train_features.pt"
CELEB_TEST_FEATURE_PATH = "/kaggle/input/datasets/vhonghoavin/deepfakebench-features/celebdfv1-color-constrast-5.pt"
FFPP_TEST_FEATURE_PATH = "/kaggle/input/datasets/vhonghoavin/deepfakebench-features/ffpp-color-constrast-5.pt"

MODEL_OUTPUT_PATH = "/kaggle/working/ufd_linear_probe.pt"

EPOCHS = 200
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE


'cuda'

## Load Features


In [3]:
def load_feature_file(path):
    payload = torch.load(path, map_location="cpu")
    features = payload["features"].float()
    labels = payload["labels"].long()

    print(path)
    print("features:", features.shape)
    print("labels:", labels.shape)
    print("label counts:", torch.bincount(labels, minlength=2).tolist())
    print("dataset:", payload.get("dataset_name", payload.get("target_dataset", "unknown")))
    print("clip:", payload.get("clip_model", "unknown"))
    print()

    return features, labels


train_feats, train_labels = load_feature_file(TRAIN_FEATURE_PATH)
celeb_test_feats, celeb_test_labels = load_feature_file(CELEB_TEST_FEATURE_PATH)
ffpp_test_feats, ffpp_test_labels = load_feature_file(FFPP_TEST_FEATURE_PATH)


/kaggle/input/datasets/vhonghoavin/deepfakebench-features/ffpp_train_features.pt
features: torch.Size([513568, 768])
labels: torch.Size([513568])
label counts: [42690, 470878]
dataset: FaceForensics++
clip: ViT-L-14/openai

/kaggle/input/datasets/vhonghoavin/deepfakebench-features/celebdfv1-color-constrast-5.pt
features: torch.Size([38312, 768])
labels: torch.Size([38312])
label counts: [5004, 33308]
dataset: Celeb-DF-v1
clip: ViT-L-14/openai

/kaggle/input/datasets/vhonghoavin/deepfakebench-features/ffpp-color-constrast-5.pt
features: torch.Size([513568, 768])
labels: torch.Size([513568])
label counts: [42690, 470878]
dataset: FaceForensics++
clip: ViT-L-14/openai



## Linear Probe


In [4]:
class LinearProbe(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(1)


clf = LinearProbe(train_feats.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(clf.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

X_train = train_feats.to(DEVICE)
y_train = train_labels.float().to(DEVICE)


In [5]:
for epoch in range(EPOCHS):
    clf.train()

    permutation = torch.randperm(len(X_train), device=DEVICE)
    total_loss = 0.0
    total_correct = 0
    total = 0

    for start in range(0, len(X_train), BATCH_SIZE):
        idx = permutation[start:start + BATCH_SIZE]
        xb = X_train[idx]
        yb = y_train[idx]

        logits = clf(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = len(xb)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        total_loss += loss.item() * batch_size
        total_correct += (preds == yb.long()).sum().item()
        total += batch_size

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={total_loss / total:.4f} | "
        f"train_acc={total_correct / total:.4f}"
    )


Epoch 1/200 | loss=0.2513 | train_acc=0.9152
Epoch 2/200 | loss=0.1983 | train_acc=0.9170
Epoch 3/200 | loss=0.1875 | train_acc=0.9176
Epoch 4/200 | loss=0.1814 | train_acc=0.9188
Epoch 5/200 | loss=0.1773 | train_acc=0.9201
Epoch 6/200 | loss=0.1743 | train_acc=0.9212
Epoch 7/200 | loss=0.1720 | train_acc=0.9222
Epoch 8/200 | loss=0.1701 | train_acc=0.9231
Epoch 9/200 | loss=0.1684 | train_acc=0.9238
Epoch 10/200 | loss=0.1670 | train_acc=0.9244
Epoch 11/200 | loss=0.1658 | train_acc=0.9250
Epoch 12/200 | loss=0.1648 | train_acc=0.9255
Epoch 13/200 | loss=0.1638 | train_acc=0.9260
Epoch 14/200 | loss=0.1629 | train_acc=0.9264
Epoch 15/200 | loss=0.1621 | train_acc=0.9269
Epoch 16/200 | loss=0.1614 | train_acc=0.9273
Epoch 17/200 | loss=0.1608 | train_acc=0.9276
Epoch 18/200 | loss=0.1602 | train_acc=0.9279
Epoch 19/200 | loss=0.1596 | train_acc=0.9281
Epoch 20/200 | loss=0.1591 | train_acc=0.9284
Epoch 21/200 | loss=0.1586 | train_acc=0.9287
Epoch 22/200 | loss=0.1581 | train_acc=0.92

## Evaluation


In [6]:
def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[eer_idx] + fnr[eer_idx]) / 2, thresholds[eer_idx]


def evaluate_scores(y_true, y_score, y_pred, name="test"):
    eer, eer_threshold = calculate_eer(y_true, y_score)
    metrics = {
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro"),
        "auc": roc_auc_score(y_true, y_score),
        "ap": average_precision_score(y_true, y_score),
        "eer": eer,
        "eer_threshold": eer_threshold,
    }

    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key}: {value}")
    print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"]))

    return metrics


@torch.no_grad()
def evaluate_probe(model, feats, labels, name="test", batch_size=4096):
    model.eval()
    scores = []

    for start in range(0, len(feats), batch_size):
        xb = feats[start:start + batch_size].to(DEVICE)
        scores.append(torch.sigmoid(model(xb)).cpu())

    y_score = torch.cat(scores).numpy()
    y_true = labels.numpy()
    y_pred = (y_score >= 0.5).astype(int)
    return evaluate_scores(y_true, y_score, y_pred, name=name)


In [7]:
train_metrics = evaluate_probe(clf, train_feats, train_labels, name="Train")
celeb_metrics = evaluate_probe(clf, celeb_test_feats, celeb_test_labels, name="Celeb-DF-v1 Test")
ffpp_metrics = evaluate_probe(clf, ffpp_test_feats, ffpp_test_labels, name="FaceForensics++ Test")



Train
acc: 0.9347077699545143
f1: 0.7338796906146711
auc: 0.9485726808818782
ap: 0.9951740228849315
eer: 0.12486229069020904
eer_threshold: 0.865460991859436
              precision    recall  f1-score   support

        REAL       0.69      0.40      0.50     42690
        FAKE       0.95      0.98      0.97    470878

    accuracy                           0.93    513568
   macro avg       0.82      0.69      0.73    513568
weighted avg       0.93      0.93      0.93    513568


Celeb-DF-v1 Test
acc: 0.8592086030486532
f1: 0.5042850958862233
auc: 0.6731335299239892
ap: 0.9281737996419033
eer: 0.3741225345651184
eer_threshold: 0.9786158800125122
              precision    recall  f1-score   support

        REAL       0.28      0.05      0.08      5004
        FAKE       0.87      0.98      0.92     33308

    accuracy                           0.86     38312
   macro avg       0.58      0.52      0.50     38312
weighted avg       0.80      0.86      0.81     38312


FaceForensics++ 

In [8]:
torch.save(clf.state_dict(), MODEL_OUTPUT_PATH)
print("saved model to:", MODEL_OUTPUT_PATH)


saved model to: /kaggle/working/ufd_linear_probe.pt


## Tip-Adapter


In [9]:
def build_tip_cache(train_feats, train_labels, num_classes=2):
    cache_keys = F.normalize(train_feats.float(), dim=-1)
    cache_values = F.one_hot(train_labels.long(), num_classes=num_classes).float()
    return cache_keys, cache_values


@torch.no_grad()
def tip_adapter_predict(
    model,
    test_feats,
    cache_keys,
    cache_values,
    alpha=0.5,
    beta=5.5,
    batch_size=512,
):
    model.eval()
    cache_keys = cache_keys.to(DEVICE)
    cache_values = cache_values.to(DEVICE)
    probs = []

    for start in tqdm(range(0, len(test_feats), batch_size)):
        feats = test_feats[start:start + batch_size].float().to(DEVICE)
        feats = F.normalize(feats, dim=-1)

        base_logits = model(feats)
        p_fake = torch.sigmoid(base_logits)
        p_base = torch.stack([1 - p_fake, p_fake], dim=1)

        similarity = feats @ cache_keys.T
        affinity = torch.exp(-beta * (1 - similarity))
        p_cache = affinity @ cache_values
        p_cache = p_cache / (p_cache.sum(dim=1, keepdim=True) + 1e-8)

        probs.append((alpha * p_cache + (1 - alpha) * p_base).cpu())

    return torch.cat(probs, dim=0)


@torch.no_grad()
def evaluate_tip_adapter(
    model,
    test_feats,
    test_labels,
    cache_keys,
    cache_values,
    name="test",
    alpha=0.5,
    beta=5.5,
    batch_size=512,
):
    probs = tip_adapter_predict(
        model=model,
        test_feats=test_feats,
        cache_keys=cache_keys,
        cache_values=cache_values,
        alpha=alpha,
        beta=beta,
        batch_size=batch_size,
    )

    print(f"alpha: {alpha} | beta: {beta}")
    metrics = evaluate_scores(
        y_true=test_labels.numpy(),
        y_score=probs[:, 1].numpy(),
        y_pred=probs.argmax(dim=1).numpy(),
        name=name,
    )
    metrics["probs"] = probs
    return metrics


cache_keys, cache_values = build_tip_cache(train_feats, train_labels)


In [10]:
celeb_tip_metrics = evaluate_tip_adapter(
    clf,
    celeb_test_feats,
    celeb_test_labels,
    cache_keys,
    cache_values,
    name="Celeb-DF-v1 Test | UFD + Tip-Adapter",
    alpha=0.5,
    beta=5.5,
)

ffpp_tip_metrics = evaluate_tip_adapter(
    clf,
    ffpp_test_feats,
    ffpp_test_labels,
    cache_keys,
    cache_values,
    name="FaceForensics++ Test | UFD + Tip-Adapter",
    alpha=0.5,
    beta=5.5,
)


100%|██████████| 75/75 [00:10<00:00,  7.40it/s]


alpha: 0.5 | beta: 5.5

Celeb-DF-v1 Test | UFD + Tip-Adapter
acc: 0.868892253079975
f1: 0.4723388450106966
auc: 0.65081062626781
ap: 0.9160036382984101
eer: 0.38530230217171285
eer_threshold: 0.9487022161483765
              precision    recall  f1-score   support

        REAL       0.40      0.01      0.01      5004
        FAKE       0.87      1.00      0.93     33308

    accuracy                           0.87     38312
   macro avg       0.64      0.50      0.47     38312
weighted avg       0.81      0.87      0.81     38312



100%|██████████| 1004/1004 [02:15<00:00,  7.41it/s]


alpha: 0.5 | beta: 5.5

FaceForensics++ Test | UFD + Tip-Adapter
acc: 0.9169321297277089
f1: 0.47917397725095795
auc: 0.8686108268834052
ap: 0.9866985488786312
eer: 0.21630613345299954
eer_threshold: 0.9532391428947449
              precision    recall  f1-score   support

        REAL       0.84      0.00      0.00     42690
        FAKE       0.92      1.00      0.96    470878

    accuracy                           0.92    513568
   macro avg       0.88      0.50      0.48    513568
weighted avg       0.91      0.92      0.88    513568



## Alpha/Beta Sweep


In [11]:
def sweep_tip_adapter(name, feats, labels, alphas=(0, 0.2, 0.5, 0.8), betas=(0, 1.5, 3.5, 5.5, 7.5)):
    rows = []

    for alpha in alphas:
        for beta in betas:
            metrics = evaluate_tip_adapter(
                clf,
                feats,
                labels,
                cache_keys,
                cache_values,
                name=f"{name} | alpha={alpha}, beta={beta}",
                alpha=alpha,
                beta=beta,
            )
            rows.append({
                "dataset": name,
                "alpha": alpha,
                "beta": beta,
                **{key: value for key, value in metrics.items() if key != "probs"},
            })

    return pd.DataFrame(rows)


# Uncomment when needed. This can be slow because each pair scans the full test set against the full train cache.
# celeb_sweep = sweep_tip_adapter("Celeb-DF-v1", celeb_test_feats, celeb_test_labels)
# ffpp_sweep = sweep_tip_adapter("FaceForensics++", ffpp_test_feats, ffpp_test_labels)
# pd.concat([celeb_sweep, ffpp_sweep], ignore_index=True)


In [12]:
celeb_sweep = sweep_tip_adapter("Celeb-DF-v1", celeb_test_feats, celeb_test_labels)
ffpp_sweep = sweep_tip_adapter("FaceForensics++", ffpp_test_feats, ffpp_test_labels)
pd.concat([celeb_sweep, ffpp_sweep], ignore_index=True)

100%|██████████| 75/75 [00:09<00:00,  7.55it/s]


alpha: 0 | beta: 0

Celeb-DF-v1 | alpha=0, beta=0
acc: 0.8592086030486532
f1: 0.5042850958862233
auc: 0.6731335149245801
ap: 0.9281737922115412
eer: 0.3741225345651184
eer_threshold: 0.9786158800125122
              precision    recall  f1-score   support

        REAL       0.28      0.05      0.08      5004
        FAKE       0.87      0.98      0.92     33308

    accuracy                           0.86     38312
   macro avg       0.58      0.52      0.50     38312
weighted avg       0.80      0.86      0.81     38312



100%|██████████| 75/75 [00:10<00:00,  7.47it/s]


alpha: 0 | beta: 1.5

Celeb-DF-v1 | alpha=0, beta=1.5
acc: 0.8592086030486532
f1: 0.5042850958862233
auc: 0.6731335149245801
ap: 0.9281737922115412
eer: 0.3741225345651184
eer_threshold: 0.9786158800125122
              precision    recall  f1-score   support

        REAL       0.28      0.05      0.08      5004
        FAKE       0.87      0.98      0.92     33308

    accuracy                           0.86     38312
   macro avg       0.58      0.52      0.50     38312
weighted avg       0.80      0.86      0.81     38312



100%|██████████| 75/75 [00:10<00:00,  7.47it/s]


...

alpha: 0.8 | beta: 5.5

FaceForensics++ | alpha=0.8, beta=5.5
acc: 0.9168756620350178
f1: 0.47831775435117824
auc: 0.8584332233837766
ap: 0.9854824311397845
eer: 0.23080423683555829
eer_threshold: 0.9327262043952942
              precision    recall  f1-score   support

        REAL       0.00      0.00      0.00     42690
        FAKE       0.92      1.00      0.96    470878

    accuracy                           0.92    513568
   macro avg       0.46      0.50      0.48    513568
weighted avg       0.84      0.92      0.88    513568



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
100%|██████████| 1004/1004 [02:14<00:00,  7.44it/

alpha: 0.8 | beta: 7.5

FaceForensics++ | alpha=0.8, beta=7.5
acc: 0.9168756620350178
f1: 0.47831775435117824
auc: 0.8551581764954207
ap: 0.9851251511536375
eer: 0.23448219253431335
eer_threshold: 0.9330945014953613
              precision    recall  f1-score   support

        REAL       0.00      0.00      0.00     42690
        FAKE       0.92      1.00      0.96    470878

    accuracy                           0.92    513568
   macro avg       0.46      0.50      0.48    513568
weighted avg       0.84      0.92      0.88    513568



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,dataset,alpha,beta,acc,f1,auc,ap,eer,eer_threshold
0,Celeb-DF-v1,0.0,0.0,0.859209,0.504285,0.673134,0.928174,0.374123,0.978616
1,Celeb-DF-v1,0.0,1.5,0.859209,0.504285,0.673134,0.928174,0.374123,0.978616
2,Celeb-DF-v1,0.0,3.5,0.859209,0.504285,0.673134,0.928174,0.374123,0.978616
3,Celeb-DF-v1,0.0,5.5,0.859209,0.504285,0.673134,0.928174,0.374123,0.978616
4,Celeb-DF-v1,0.0,7.5,0.859209,0.504285,0.673134,0.928174,0.374123,0.978616
5,Celeb-DF-v1,0.2,0.0,0.862341,0.493644,0.673134,0.928174,0.374123,0.966268
6,Celeb-DF-v1,0.2,1.5,0.862315,0.493299,0.671736,0.926464,0.374252,0.966501
7,Celeb-DF-v1,0.2,3.5,0.862289,0.493120,0.669479,0.924805,0.375938,0.966760
8,Celeb-DF-v1,0.2,5.5,0.862262,0.493107,0.667164,0.923406,0.378042,0.966995
9,Celeb-DF-v1,0.2,7.5,0.862236,0.492928,0.664877,0.922147,0.379082,0.967122
